# Experimentos REE — Parámetros óptimos del TFM
**Fuente**: Javier Jiménez Arenas, UGR 2025 — Tablas 6.3 (24h) y 6.4 (48h)  
**Dataset**: REE (demanda eléctrica nacional española, intervalos 10 min)  
**Split**: 70% train / 30% test (cronológico). Validación = último 30% del train.  
**Scaler**: MinMax ajustado sobre el 70% completo del train (igual que el TFM).  
**Modelos**: MLP, Elman, LSTM, GRU, Transformer × 2 ventanas (144 y 288 pasos)

## 0. Entorno — Colab / Kaggle / ELECTRA / local
Esta celda detecta el entorno y configura automáticamente los paths.  
**Colab / Kaggle con repo privado**: reemplaza la URL con `https://<TOKEN>@github.com/erfundoraUGR/LabExperimental.git`  
**Kaggle**: activa *Internet* y *GPU* en Settings antes de ejecutar.

In [ ]:
import sys, os, subprocess

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')

REPO = 'https://github.com/erfundoraUGR/LabExperimental.git'

def _clone(repo, folder):
    """Clona repo en folder; muestra salida y lanza error si falla."""
    result = subprocess.run(
        ['git', 'clone', repo, folder],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'git clone falló (código {result.returncode}). '
                           f'Verifica que Internet esté activado en Settings.')

if IN_COLAB:
    FOLDER = '/content/LabExperimental'
    if not os.path.exists(FOLDER):
        _clone(REPO, FOLDER)
    os.chdir(FOLDER)
    print(f'Colab — directorio: {os.getcwd()}')

elif IN_KAGGLE:
    FOLDER = '/kaggle/working/LabExperimental'
    if not os.path.exists(FOLDER):
        _clone(REPO, FOLDER)
    os.chdir(FOLDER)
    print(f'Kaggle — directorio: {os.getcwd()}')
    print('Resultados guardados en /kaggle/working/ → accesibles en la pestaña Output')

else:
    # ELECTRA o local: el notebook ya está en la raíz del proyecto
    print(f'Entorno local/cluster — directorio: {os.getcwd()}')

sys.path.insert(0, os.path.abspath('.'))

## 1. Imports y dispositivo

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display

from core import Config, set_seed, build_model, train, evaluate
from core.data import load_data, preprocess, create_sequences
from core.utils import create_experiment_folder, save_config, log_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Verificación del dataset

In [ ]:
DATA_PATH = 'dataset/combined_data_2009_2019.csv'

df_raw = load_data(DATA_PATH)
print(f'Rango temporal : {df_raw.index.min()}  →  {df_raw.index.max()}')
print(f'Total registros: {len(df_raw):,}')
print(f'Columnas       : {list(df_raw.columns)}')

n = len(df_raw)
train_end = int(n * 0.70)
val_start = int(train_end * 0.70)
print(f'\nSplit TFM (sobre {n:,} registros):')
print(f'  subtrain : {val_start:,}  ({val_start/n*100:.1f}%)  →  hasta {df_raw.index[val_start]}')
print(f'  val      : {train_end - val_start:,}  ({(train_end-val_start)/n*100:.1f}%)  →  hasta {df_raw.index[train_end]}')
print(f'  test     : {n - train_end:,}  ({(n-train_end)/n*100:.1f}%)  →  hasta {df_raw.index[-1]}')

display(df_raw.head())

plt.figure(figsize=(14, 3))
df_raw['dem'].plot(title='Demanda eléctrica REE (MW)', alpha=0.6)
plt.axvline(df_raw.index[train_end], color='red',    linestyle='--', label='inicio test (70%)')
plt.axvline(df_raw.index[val_start], color='orange', linestyle='--', label='inicio val (49%)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Configuraciones óptimas del TFM

In [ ]:
# (model_type, window, batch_size, lr, hidden_sizes, num_heads, num_layers_enc)
# Transformer: hidden_sizes[0]=d_model, [1:]=capas FC. d_model=50→52 (múltiplo de heads=4).

TFM_CONFIGS = [
    ('MLP',         144, 32, 0.0002844, [77, 57],           None, None),
    ('ELMAN',       144, 64, 0.0001589, [97, 44, 48],       None, None),
    ('LSTM',        144, 32, 0.0017554, [86, 16, 2],        None, None),
    ('GRU',         144, 16, 0.0008212, [82, 46],           None, None),
    ('TRANSFORMER', 144, 32, 0.0005801, [52, 71, 100, 85],  4,    1   ),

    ('MLP',         288, 32, 0.0003431, [81, 32],           None, None),
    ('ELMAN',       288, 32, 0.0002024, [19, 100],          None, None),
    ('LSTM',        288, 32, 0.0009840, [86, 18],           None, None),
    ('GRU',         288, 16, 0.0008951, [65, 39],           None, None),
    ('TRANSFORMER', 288, 16, 0.0001083, [92, 47, 75],       2,    1   ),
]

# Métricas de referencia del TFM (Tablas 6.16 y 6.17) — conjunto de test
TFM_REFERENCE = {
    ('MLP',         144): dict(RMSE=365.80, MAE=206.25, MAPE=0.7288, R2=0.9935),
    ('ELMAN',       144): dict(RMSE=370.58, MAE=210.51, MAPE=0.7399, R2=0.9935),
    ('LSTM',        144): dict(RMSE=336.17, MAE=182.89, MAPE=0.6445, R2=0.9945),
    ('GRU',         144): dict(RMSE=329.87, MAE=172.07, MAPE=0.6036, R2=0.9948),
    ('TRANSFORMER', 144): dict(RMSE=329.01, MAE=170.86, MAPE=0.6054, R2=0.9949),
    ('MLP',         288): dict(RMSE=342.68, MAE=181.12, MAPE=0.6383, R2=0.9944),
    ('ELMAN',       288): dict(RMSE=390.25, MAE=230.62, MAPE=0.8132, R2=0.9927),
    ('LSTM',        288): dict(RMSE=347.41, MAE=189.40, MAPE=0.6672, R2=0.9941),
    ('GRU',         288): dict(RMSE=347.77, MAE=191.88, MAPE=0.6714, R2=0.9942),
    ('TRANSFORMER', 288): dict(RMSE=331.30, MAE=177.47, MAPE=0.6225, R2=0.9945),
}

print(f'{len(TFM_CONFIGS)} experimentos configurados.')

## 4. Pipeline de datos y función de experimento

In [ ]:
def build_loaders_tfm(df, window, batch_size):
    """Split y normalización exactos del TFM.
    - 70% train / 30% test (cronológico, sin shuffle)
    - Validación = último 30% del bloque train
    - MinMaxScaler ajustado sobre el 70% completo (subtrain + val)
    """
    n = len(df)
    train_end  = int(n * 0.70)
    full_train = df[:train_end]
    test_df    = df[train_end:]

    scaler = MinMaxScaler()
    scaler.fit(full_train)
    full_train_s = scaler.transform(full_train)
    test_s       = scaler.transform(test_df)

    val_start = int(len(full_train_s) * 0.70)
    train_s   = full_train_s[:val_start]
    val_s     = full_train_s[val_start:]

    X_tr, y_tr = create_sequences(train_s, window, 1)
    X_va, y_va = create_sequences(val_s,   window, 1)
    X_te, y_te = create_sequences(test_s,  window, 1)

    def make_loader(X, y, bs):
        return DataLoader(
            TensorDataset(torch.tensor(X).float(), torch.tensor(y).float()),
            batch_size=bs, shuffle=False)

    return (make_loader(X_tr, y_tr, batch_size),
            make_loader(X_va, y_va, batch_size),
            make_loader(X_te, y_te, batch_size),
            scaler)


def run_experiment(model_type, window, batch_size, lr, hidden_sizes,
                   num_heads=None, num_layers_enc=None, seed=42):
    set_seed(seed)

    cfg = Config()
    cfg.DEVICE       = DEVICE
    cfg.SEED         = seed
    cfg.MODEL_TYPE   = model_type
    cfg.WINDOW_SIZE  = window
    cfg.BATCH_SIZE   = batch_size
    cfg.LR           = lr
    cfg.EPOCHS       = 20
    cfg.EARLY_STOPPING_PATIENCE  = 5
    cfg.EARLY_STOPPING_MIN_DELTA = 1e-6

    setattr(cfg, f'{model_type}_HIDDEN_SIZES', hidden_sizes)
    if num_heads is not None:
        cfg.TRANSFORMER_NUM_HEADS  = num_heads
    if num_layers_enc is not None:
        cfg.TRANSFORMER_NUM_LAYERS = num_layers_enc

    df = load_data(DATA_PATH)
    df = preprocess(df, cfg)
    train_loader, val_loader, test_loader, scaler = build_loaders_tfm(df, window, batch_size)

    exp_path = create_experiment_folder()
    model    = build_model(cfg)
    history  = train(model, train_loader, val_loader, cfg, exp_path)

    model.load_state_dict(torch.load(f'{exp_path}/models/best_model.pt',
                                     map_location=DEVICE))

    # Evaluar en los tres conjuntos
    train_metrics = evaluate(model, train_loader, cfg, scaler)
    val_metrics   = evaluate(model, val_loader,   cfg, scaler)
    test_metrics  = evaluate(model, test_loader,  cfg, scaler)

    all_metrics = {'train': train_metrics, 'val': val_metrics, 'test': test_metrics}
    save_config(cfg, exp_path)
    log_metrics(exp_path, all_metrics, cfg)

    return all_metrics, history, exp_path


results = {}  # (model_type, window) -> {'train': {...}, 'val': {...}, 'test': {...}}

def run_and_store(model_type, window, batch_size, lr, hidden_sizes,
                  num_heads=None, num_layers_enc=None):
    print(f'\n>>> {model_type}-{window}  hidden={hidden_sizes}  bs={batch_size}  lr={lr}')
    all_metrics, history, path = run_experiment(
        model_type, window, batch_size, lr, hidden_sizes, num_heads, num_layers_enc)
    results[(model_type, window)] = all_metrics
    ref = TFM_REFERENCE.get((model_type, window), {})

    print(f'  {"Conjunto":<8}  {"RMSE":>8}  {"MAE":>8}  {"MAPE":>8}  {"R²":>8}')
    print(f'  {"-"*52}')
    for split_name in ['train', 'val', 'test']:
        m = all_metrics[split_name]
        print(f'  {split_name:<8}  {m["RMSE"]:>8.2f}  {m["MAE"]:>8.2f}  {m["MAPE"]:>8.4f}  {m["R2"]:>8.4f}')
    if ref:
        print(f'  {"TFM ref":<8}  {ref["RMSE"]:>8.2f}  {ref["MAE"]:>8.2f}  {ref["MAPE"]:>8.4f}  {ref["R2"]:>8.4f}')
    print(f'  Guardado en: {path}')
    return all_metrics


print('Pipeline listo.')

## 5. Ejecución — 24h (WINDOW = 144 pasos)

### 5.1 MLP — 24h

In [ ]:
run_and_store('MLP', 144, 32, 0.0002844, [77, 57])

### 5.2 Elman — 24h

In [ ]:
run_and_store('ELMAN', 144, 64, 0.0001589, [97, 44, 48])

### 5.3 LSTM — 24h

In [ ]:
run_and_store('LSTM', 144, 32, 0.0017554, [86, 16, 2])

### 5.4 GRU — 24h

In [ ]:
run_and_store('GRU', 144, 16, 0.0008212, [82, 46])

### 5.5 Transformer — 24h

In [ ]:
run_and_store('TRANSFORMER', 144, 32, 0.0005801, [52, 71, 100, 85], num_heads=4, num_layers_enc=1)

## 6. Ejecución — 48h (WINDOW = 288 pasos)

### 6.1 MLP — 48h

In [ ]:
run_and_store('MLP', 288, 32, 0.0003431, [81, 32])

### 6.2 Elman — 48h

In [ ]:
run_and_store('ELMAN', 288, 32, 0.0002024, [19, 100])

### 6.3 LSTM — 48h

In [ ]:
run_and_store('LSTM', 288, 32, 0.0009840, [86, 18])

### 6.4 GRU — 48h

In [ ]:
run_and_store('GRU', 288, 16, 0.0008951, [65, 39])

### 6.5 Transformer — 48h

In [ ]:
run_and_store('TRANSFORMER', 288, 16, 0.0001083, [92, 47, 75], num_heads=2, num_layers_enc=1)

## 7. Tabla comparativa — train / val / test vs TFM

In [ ]:
rows = []
for model_type, window, *_ in TFM_CONFIGS:
    key = (model_type, window)
    if key not in results:
        continue
    m   = results[key]
    ref = TFM_REFERENCE.get(key, {})
    rows.append({
        'Modelo':           model_type,
        'Ventana':          f'{window} ({window//144*24}h)',
        'RMSE train':       round(m['train']['RMSE'], 2),
        'RMSE val':         round(m['val']['RMSE'],   2),
        'RMSE test':        round(m['test']['RMSE'],  2),
        'RMSE TFM':         ref.get('RMSE', '-'),
        'Δ RMSE test-TFM':  round(m['test']['RMSE'] - ref['RMSE'], 2) if ref else '-',
        'MAE test':         round(m['test']['MAE'],  2),
        'MAE TFM':          ref.get('MAE', '-'),
        'MAPE test':        round(m['test']['MAPE'], 4),
        'MAPE TFM':         ref.get('MAPE', '-'),
        'R² test':          round(m['test']['R2'],   4),
        'R² TFM':           ref.get('R2', '-'),
    })

df_results = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
display(df_results)

os.makedirs('experiments', exist_ok=True)
df_results.to_csv('experiments/comparativa_tfm_ree.csv', index=False)
print('Tabla guardada en experiments/comparativa_tfm_ree.csv')

# Descarga automática solo en Colab (Kaggle → pestaña Output, ELECTRA → experiments/)
if IN_COLAB:
    from google.colab import files
    files.download('experiments/comparativa_tfm_ree.csv')

## 8. Análisis de overfitting — RMSE train vs val vs test

In [ ]:
if results:
    for window in [144, 288]:
        keys = [(m, w) for (m, w) in results if w == window]
        if not keys:
            continue

        models  = [k[0] for k in keys]
        rmse_tr = [results[k]['train']['RMSE'] for k in keys]
        rmse_va = [results[k]['val']['RMSE']   for k in keys]
        rmse_te = [results[k]['test']['RMSE']  for k in keys]

        x = range(len(models))
        w = 0.25
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar([i - w for i in x], rmse_tr, w, label='Train', color='#4CAF50')
        ax.bar([i     for i in x], rmse_va, w, label='Val',   color='#FF9800')
        ax.bar([i + w for i in x], rmse_te, w, label='Test',  color='#2196F3')
        ax.set_xticks(list(x))
        ax.set_xticklabels(models, rotation=15)
        ax.set_ylabel('RMSE (MW)')
        ax.set_title(f'RMSE por conjunto — Ventana {window} ({window//144*24}h lag)')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'experiments/overfitting_{window}.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Guardado: experiments/overfitting_{window}.png')

        if IN_COLAB:
            from google.colab import files
            files.download(f'experiments/overfitting_{window}.png')

## 9. Gráfica RMSE test — Nuestro vs TFM

In [ ]:
if not df_results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, window in zip(axes, [144, 288]):
        label_w = f'{window} ({window//144*24}h)'
        subset  = df_results[df_results['Ventana'] == label_w]
        if subset.empty:
            continue
        x = range(len(subset))
        w = 0.35
        b1 = ax.bar([i - w/2 for i in x], subset['RMSE test'],
                    w, label='Nuestro (test)', color='steelblue')
        b2 = ax.bar([i + w/2 for i in x], subset['RMSE TFM'],
                    w, label='TFM (ref)', color='coral', alpha=0.75)
        ax.set_xticks(list(x))
        ax.set_xticklabels(subset['Modelo'], rotation=15)
        ax.set_ylabel('RMSE (MW)')
        ax.set_title(f'RMSE — Ventana {window} ({window//144*24}h lag)')
        ax.legend()
        ax.bar_label(b1, fmt='%.0f', padding=2, fontsize=8)
        ax.bar_label(b2, fmt='%.0f', padding=2, fontsize=8)

    plt.suptitle('Comparativa RMSE test: Nuestro proyecto vs TFM (REE dataset)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('experiments/comparativa_rmse.png', dpi=150, bbox_inches='tight')
    plt.show()

    if IN_COLAB:
        from google.colab import files
        files.download('experiments/comparativa_rmse.png')